In [1]:
!pip install geopandas pyogrio shapely ipywidgets -q


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
from ipywidgets import FileUpload

upload_dir = Path.cwd() / 'uploaded_data'
upload_dir.mkdir(exist_ok=True)

uploader = FileUpload(accept='.shp,.dbf,.shx,.prj,.cpg,.kmz,.geojson,.csv', multiple=True)
display(uploader)

print(f'Choose your local files and then run the next cell to save them into: {upload_dir}')

FileUpload(value=(), accept='.shp,.dbf,.shx,.prj,.cpg,.kmz,.geojson,.csv', description='Upload', multiple=True…

Choose your local files and then run the next cell to save them into: d:\User\isuru.sanjana\Downloads\LS-LRSI_Passara_Submission\code\uploaded_data


In [5]:
import os
import geopandas as gpd
from pathlib import Path

# Restore missing/corrupt .shx index files when GDAL can rebuild them from the .shp
os.environ.setdefault('SHAPE_RESTORE_SHX', 'YES')

upload_dir = Path.cwd() / 'uploaded_data'
if not upload_dir.exists():
    raise FileNotFoundError('No uploaded files were found. Use the uploader above and then rerun this cell.')

shp_path = next(upload_dir.glob('*.shp'), None)
if shp_path is None:
    raise FileNotFoundError('No .shp file was found in uploaded_data. Make sure the shapefile components (.shp, .dbf, .shx, .prj) were uploaded together.')

gdf = gpd.read_file(shp_path)

# GeoJSON
gdf.to_file('shp_landslides.geojson', driver='GeoJSON')

# CSV
gdf['centroid_lon'] = gdf.geometry.centroid.x
gdf['centroid_lat'] = gdf.geometry.centroid.y
gdf['geometry_wkt'] = gdf.geometry.apply(lambda g: g.wkt)
gdf.drop(columns='geometry').to_csv('shp_landslides.csv', index=False)

print(gdf.shape)
print('Saved:', 'shp_landslides.geojson', 'and', 'shp_landslides.csv')

c:\Users\isuru.sanjana\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyogrio\geopandas.py:948: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


(4225, 4)
Saved: shp_landslides.geojson and shp_landslides.csv


In [6]:
upload_dir = Path.cwd() / 'uploaded_data'
kmz_path = next(upload_dir.glob('*.kmz'), None)
if kmz_path is None:
    print('No .kmz file was found in uploaded_data; skipping KMZ conversion.')
else:
    kmz = gpd.read_file(kmz_path)
    kmz.to_file('kmz_landslides.geojson', driver='GeoJSON')

    kmz_csv = kmz.copy()
    kmz_csv['centroid_lon'] = kmz_csv.geometry.centroid.x
    kmz_csv['centroid_lat'] = kmz_csv.geometry.centroid.y
    kmz_csv['geometry_wkt'] = kmz_csv.geometry.apply(lambda g: g.wkt)
    kmz_csv.drop(columns='geometry').to_csv('kmz_landslides.csv', index=False)

    print(kmz.shape)
    print('Saved:', 'kmz_landslides.geojson', 'and', 'kmz_landslides.csv')

No .kmz file was found in uploaded_data; skipping KMZ conversion.


In [7]:
output_files = [
    'shp_landslides.geojson',
    'shp_landslides.csv',
    'kmz_landslides.geojson',
    'kmz_landslides.csv',
]

for file_name in output_files:
    path = Path(file_name)
    print(f'{file_name}:', 'exists' if path.exists() else 'not created yet')

shp_landslides.geojson: exists
shp_landslides.csv: exists
kmz_landslides.geojson: not created yet
kmz_landslides.csv: not created yet
